# Project 1: Company-Name-Only Baseline
Each company receives one request containing its name, the seven criteria, and output instructions. No policy text, URLs, web tools, or anti-hallucination instruction are supplied.

Run cells in order with the same `Project 1 Criteria.xlsx`. Set `RUN_BASELINE = True` to run the experiment. Running the experiment cell again makes new requests.

Results and the original classification metrics appear below the cells. A separate CSV contains one row per company/right for manual review of quotations and explanations. Failed or incomplete responses are reported separately, never converted to Unsure.

In [ ]:
%pip install -q openai pandas openpyxl pydantic

## 1. Setup

In [ ]:
import os, json, getpass
from pathlib import Path
from typing import Literal
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field
from openai import OpenAI
from IPython.display import display
from google.colab import userdata

CRITERIA_PATH = Path("Criteria.xlsx")
RUN_BASELINE = True
OUTPUT_DIR = Path("standard_baseline_results")
OUTPUT_DIR.mkdir(exist_ok=True)
STATUSES = ["Met", "Not Met", "Unsure", "Not Applicable"]
api_key = userdata.get("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL", "@30875-fall26-openai/gpt-5-nano")
client = OpenAI(api_key=api_key, base_url="https://api.portkey.ai/v1", timeout=90.0, max_retries=1)


print("Model:", MODEL)
print("Client configured; the API key was not displayed.")

Model: @30875-fall26-openai/gpt-5-nano
Client configured; the API key was not displayed.


## 2. Load the criteria and golden set

In [ ]:
if not CRITERIA_PATH.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Place Project 1 Criteria.xlsx beside the notebook or update CRITERIA_PATH.")
    uploaded = files.upload()
    matches = [Path(name) for name in uploaded if "criteria" in name.lower() and name.endswith(".xlsx")]
    if len(matches) != 1:
        raise ValueError("Set CRITERIA_PATH to the exact uploaded workbook and rerun this cell.")
    CRITERIA_PATH = matches[0]

criteria_df = pd.read_excel(CRITERIA_PATH, sheet_name="Criteria", dtype=str).dropna(subset=["Right ID"])
golden_df = pd.read_excel(CRITERIA_PATH, sheet_name="Golden Set", dtype=str).dropna(subset=["Registration Number"])
for col in ["Right ID", "Core Right", "What to look up on the page", "Statute Location"]:
    assert criteria_df[col].notna().all(), f"Missing {col}"
    criteria_df[col] = criteria_df[col].str.strip()
RIGHT_IDS = criteria_df["Right ID"].tolist()
assert len(RIGHT_IDS) == 7 and set(RIGHT_IDS) == {f"r{i}" for i in range(1, 8)}
for col in ["Registration Number", "Legal Name", *RIGHT_IDS]:
    assert golden_df[col].notna().all(), f"Missing {col}"
    golden_df[col] = golden_df[col].str.strip()
assert not golden_df["Registration Number"].duplicated().any()
assert golden_df["Legal Name"].ne("").all()
assert golden_df[RIGHT_IDS].isin(STATUSES).all().all(), "Unexpected golden-set status"

criteria = [{"right_id": row["Right ID"], "core_right": row["Core Right"],
             "criterion": row["What to look up on the page"], "statute_location": row["Statute Location"]}
            for _, row in criteria_df.iterrows()]
print(f"{len(golden_df)} companies × 7 rights; {len(golden_df)} model requests.")
display(criteria_df[["Right ID", "Core Right", "What to look up on the page"]])
RUBRIC = {r["right_id"]: {"right_id": r["right_id"], "core_right": r["core_right"],
           "what_to_look_up": r["criterion"], "statute_location": r["statute_location"]}
          for r in criteria}


20 companies × 7 rights; 20 model requests.


,Right ID,Core Right,What to look up on the page
0,r1,Confirmation and access,Instructions for asking whether the company pr...
1,r2,Correction,Instructions for correcting inaccurate persona...
2,r3,Deletion,Instructions for deleting personal data provid...
3,r4,Data portability,"Instructions for obtaining a portable, readily..."
4,r5,Opt out of targeted advertising,Instructions for opting out of processing pers...
5,r6,Opt out of sale,Instructions for opting out of the sale of per...
6,r7,Opt out of significant profiling,Instructions for opting out of profiling used ...


## 3. Original baseline approach, without page text
The helper makes one `responses.parse()` request and returns the parsed model output, tokens, and error. There are no helper or SDK retries. The output schema requires seven entries and a separate check verifies each right occurs exactly once. This controls output structure without prescribing a status.

In [ ]:
class BaselineItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    right_id: Literal["r1", "r2", "r3", "r4", "r5", "r6", "r7"]
    status: Literal["Met", "Not Met", "Unsure", "Not Applicable"]
    evidence_quote: str
    explanation: str

class BaselineReview(BaseModel):
    model_config = ConfigDict(extra="forbid")
    findings: list[BaselineItem] = Field(min_length=7, max_length=7)

BASELINE_INSTRUCTIONS = (
    "Review the named company's privacy policy against the seven "
    "Texas consumer-rights criteria provided. "
    "For EVERY right, return one finding containing its right_id, "
    "status, evidence_quote, and explanation. "
    "Use one of these status labels: Met, Not Met, Unsure, or Not Applicable. "
    "The evidence_quote should be a short exact quotation "
    "from the policy."
)

def llm_parse(instructions, user_input, schema):
    tokens = 0
    try:
        response = client.responses.parse(
            model=MODEL, instructions=instructions,
            input=user_input, text_format=schema,
        )
        tokens = response.usage.total_tokens if response.usage else 0
        parsed = response.output_parsed
        if response.status != "completed" or parsed is None:
            raise ValueError("Response incomplete or no parsable output")
        return {"parsed": parsed, "tokens": tokens, "error": None}
    except Exception as error:
        return {"parsed": None, "tokens": tokens,
                "error": f"{type(error).__name__}: {str(error)[:300]}"}

def baseline_input(entry):
    rubric_text = "\n".join(
        f"{r['right_id']}: {r['core_right']}. "
        f"{r['what_to_look_up']} ({r['statute_location']})"
        for r in RUBRIC.values()
    )
    return f"COMPANY: {entry['name']}\n\nRUBRIC:\n{rubric_text}"

def run_baseline(entry):
    result = llm_parse(BASELINE_INSTRUCTIONS, baseline_input(entry), BaselineReview)
    predictions = {}
    error = result["error"]
    parsed = result["parsed"]
    if parsed is not None:
        ids = [item.right_id for item in parsed.findings]
        if len(ids) != len(RUBRIC) or set(ids) != set(RUBRIC):
            error = "Expected exactly one finding for each right r1–r7."
        else:
            predictions = {
                item.right_id: {"status": item.status, "quote": item.evidence_quote,
                                "explanation": item.explanation}
                for item in parsed.findings
            }
    return {"predictions": predictions, "tokens": result["tokens"], "error": error,
            "model_output": parsed.model_dump() if parsed is not None else None}

print(BASELINE_INSTRUCTIONS)
print(baseline_input({"name": golden_df.iloc[0]["Legal Name"]}))

Review the named company's privacy policy against the seven Texas consumer-rights criteria provided. For EVERY right, return one finding containing its right_id, status, evidence_quote, and explanation. Use one of these status labels: Met, Not Met, Unsure, or Not Applicable. The evidence_quote should be a short exact quotation from the policy.
COMPANY: PHANTOM DATA LLC

RUBRIC:
r1: Confirmation and access. Instructions for asking whether the company processes the consumer’s personal data and accessing that data. (§541.051(b)(1))
r2: Correction. Instructions for correcting inaccurate personal data, considering its nature and processing purposes. (§541.051(b)(2))
r3: Deletion. Instructions for deleting personal data provided by the consumer or obtained about them from other sources. (§541.051(b)(3))
r4: Data portability. Instructions for obtaining a portable, readily usable copy of personal data previously provided by the consumer, when digitally available and technically feasible. (§541

## 4. Run one call per company
Each result is saved to readable `baseline_responses.json`, with the parsed findings, token count, and any error. Successful findings are also displayed and exported below.

In [ ]:
results, errors, raw_responses = [], [], []
if RUN_BASELINE:
    for _, company in golden_df.iterrows():
        entry = {"name": company["Legal Name"], "reg_no": company["Registration Number"]}
        baseline = run_baseline(entry)
        raw_responses.append({"company_name": entry["name"], "reg_no": entry["reg_no"],
                              "model": MODEL, **baseline})
        (OUTPUT_DIR / "baseline_responses.json").write_text(
            json.dumps(raw_responses, indent=2, ensure_ascii=False), encoding="utf-8")
        if not baseline["predictions"]:
            errors.append({"company_name": entry["name"], "reg_no": entry["reg_no"],
                           "error": baseline["error"]})
            print(f"Failed: {entry['name']} — {baseline['error']}")
            continue
        for rid in RIGHT_IDS:
            f = baseline["predictions"][rid]
            results.append({"company_name": entry["name"], "reg_no": entry["reg_no"],
                "right": rid, "gold": company[rid], "baseline": f["status"],
                "evidence_quote": f["quote"], "explanation": f["explanation"]})
        print(f"Completed: {entry['name']}")
else:
    print("Set RUN_BASELINE = True in Setup and rerun this cell to make model requests.")

eval_df = pd.DataFrame(results, columns=["company_name", "reg_no", "right", "gold", "baseline", "evidence_quote", "explanation"])
errors_df = pd.DataFrame(errors, columns=["company_name", "reg_no", "error"])

Completed: PHANTOM DATA LLC
Completed: Torus Voter Systems, Inc.
Completed: The Segerdahl LLC d/b/a sg360°
Completed: Criteo Corp.
Completed: Tracerfy LLC
Completed: 5x5 US, LLC
Completed: BLUE DATA, LLC
Completed: Semasio, Inc.
Completed: DT Client Services LLC
Completed: ATMOSPHERIK INC.
Completed: My Lead Hawk, LLC
Completed: Intentsify, LLC
Completed: Catchlight Insights LLC
Completed: Forian Holdings, LLC
Completed: DMS Purchaser, LLC
Completed: Zoom Communications, Inc.
Completed: Common Room, Inc.
Completed: Kloudend, Inc.
Completed: Sovra AI, Inc.
Completed: Sprout Social, Inc.


## 5. Classification metrics
**Not Met in the golden set is the positive class.**

- **Precision:** TP / (TP + FP). **Recall:** TP / (TP + FN).
- **FPR:** FP / (FP + TN). **Exact agreement:** proportion of exact matches across all four labels.
- **Unsure rate:** proportion of model answers labeled Unsure.

Metrics use successfully completed companies. Failed companies are reported separately and excluded, so report completion counts alongside the scores. Undefined ratios appear as NaN. These scores measure agreement with the golden set, not whether quotations or explanations were hallucinated.

In [ ]:
def violation_metrics(df, column, flag_unsure=False):
    flagged = (df[column] == "Not Met") | ((df[column] == "Unsure") if flag_unsure else False)
    positive = df["gold"] == "Not Met"
    tp, fp = int((flagged & positive).sum()), int((flagged & ~positive).sum())
    fn, tn = int((~flagged & positive).sum()), int((~flagged & ~positive).sum())
    ratio = lambda a, b: round(a / b, 3) if b else float("nan")
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": ratio(tp, tp + fp), "recall": ratio(tp, tp + fn), "FPR": ratio(fp, fp + tn)}

metrics_df = pd.DataFrame()
if not eval_df.empty:
    metric_rows = []
    for scoring, flag in [("Unsure not flagged", False), ("Unsure flagged", True)]:
        metric_rows.append({"system": "standard baseline", "scoring": scoring,
            **violation_metrics(eval_df, "baseline", flag),
            "exact_agreement": round(float((eval_df.baseline == eval_df.gold).mean()), 3),
            "unsure_rate": round(float((eval_df.baseline == "Unsure").mean()), 3)})
    metrics_df = pd.DataFrame(metric_rows)
    print(f"Completed companies: {eval_df.reg_no.nunique()}/{len(golden_df)}; failed: {len(errors_df)}")
    display(metrics_df)
    display(eval_df)
else:
    print("No successful model results to score.")
if not errors_df.empty:
    display(errors_df)

Completed companies: 20/20; failed: 0


,system,scoring,TP,FP,FN,TN,precision,recall,FPR,exact_agreement,unsure_rate
0,standard baseline,Unsure not flagged,4,73,21,42,0.052,0.16,0.635,0.029,0.0
1,standard baseline,Unsure flagged,4,73,21,42,0.052,0.16,0.635,0.029,0.0


,company_name,reg_no,right,gold,baseline,evidence_quote,explanation
0,PHANTOM DATA LLC,20260190,r1,Met,Not Met,,I don’t yet have Phantom Data LLC’s privacy po...
1,PHANTOM DATA LLC,20260190,r2,Met,Not Met,,Awaiting policy content to check for instructi...
2,PHANTOM DATA LLC,20260190,r3,Met,Not Met,,Awaiting policy content to check for instructi...
3,PHANTOM DATA LLC,20260190,r4,Met,Not Met,,Awaiting policy content to check for data port...
4,PHANTOM DATA LLC,20260190,r5,Met,Not Met,,Awaiting policy content to check for opt-out i...
...,...,...,...,...,...,...,...
135,"Sprout Social, Inc.",20260171,r3,Met,Not Met,,Awaiting the policy text to verify whether the...
136,"Sprout Social, Inc.",20260171,r4,Not Met,Not Met,,Awaiting the policy text to check for data por...
137,"Sprout Social, Inc.",20260171,r5,Met,Not Met,,Awaiting the policy text to determine if there...
138,"Sprout Social, Inc.",20260171,r6,Met,Not Met,,Awaiting the policy text to identify whether t...


## 6. Save CSV outputs for evaluation and human review
`standard_baseline_human_review.csv` has **seven rows per successfully evaluated company**, one for each right r1–r7. It includes the company name, right, model status, evidence quote, and explanation, plus blank columns for your manual review. A wrong explanation does not necessarily mean a fabricated quote; assess these separately.

The other CSVs contain predictions with golden labels, classification metrics, and any errors.

In [ ]:
if not eval_df.empty:
    review_df = eval_df[["company_name", "right", "baseline", "evidence_quote", "explanation"]].rename(columns={"baseline": "model_status"}).copy()
    review_df["fabricated_quote"] = ""
    review_df["wrong_explanation"] = ""
    review_df["review_notes"] = ""
    review_df.to_csv(OUTPUT_DIR / "standard_baseline_human_review.csv", index=False)
    eval_df.to_csv(OUTPUT_DIR / "standard_baseline_predictions.csv", index=False)
    metrics_df.to_csv(OUTPUT_DIR / "standard_baseline_metrics.csv", index=False)
    display(review_df)
    print("CSV files saved in:", OUTPUT_DIR.resolve())
if raw_responses:
    errors_df.to_csv(OUTPUT_DIR / "standard_baseline_errors.csv", index=False)

,company_name,right,model_status,evidence_quote,explanation,fabricated_quote,wrong_explanation,review_notes
0,PHANTOM DATA LLC,r1,Not Met,,I don’t yet have Phantom Data LLC’s privacy po...,,,
1,PHANTOM DATA LLC,r2,Not Met,,Awaiting policy content to check for instructi...,,,
2,PHANTOM DATA LLC,r3,Not Met,,Awaiting policy content to check for instructi...,,,
3,PHANTOM DATA LLC,r4,Not Met,,Awaiting policy content to check for data port...,,,
4,PHANTOM DATA LLC,r5,Not Met,,Awaiting policy content to check for opt-out i...,,,
...,...,...,...,...,...,...,...,...
135,"Sprout Social, Inc.",r3,Not Met,,Awaiting the policy text to verify whether the...,,,
136,"Sprout Social, Inc.",r4,Not Met,,Awaiting the policy text to check for data por...,,,
137,"Sprout Social, Inc.",r5,Not Met,,Awaiting the policy text to determine if there...,,,
138,"Sprout Social, Inc.",r6,Not Met,,Awaiting the policy text to identify whether t...,,,


CSV files saved in: /content/standard_baseline_results


# Key Findings
The company-name-only baseline completed all 140 evaluations across 20 companies without execution errors, but performed poorly against the golden set. It produced 4 true positives, 73 false positives, 21 false negatives, and 42 true negatives, resulting in 5.2% precision, 16% recall, and 2.9% exact agreement. These results indicate that company names and evaluation criteria alone were insufficient for reliable policy classification in this run.

The most significant finding was a mismatch between the model's explanations and its assigned labels. Although its explanations repeatedly acknowledged that policy text was unavailable, the model labeled 77 findings Not Met and 63 Not Applicable, with no Unsure responses. Every company received the same label across all seven rights. The model therefore recognized missing evidence in its explanations but failed to represent that limitation appropriately in its classifications. Even the four true positives were unsupported by policy evidence and should not be interpreted as demonstrated gap detection.

The results showed limited evidence of hallucinated policy language. Of the 140 evidence fields, 133 were empty. The remaining seven, all associated with Tracerfy, contained “Cannot access the policy at the moment.” This was model commentary incorrectly placed in the quotation field rather than supporting policy evidence. The primary observed failure was therefore unsupported status assignment and inconsistent output, rather than the invention of detailed policy provisions.

Overall, this baseline demonstrates that acknowledging insufficient information does not necessarily produce an appropriate abstention